# Week 16: Agent Studio II: an evaluation harness

Machine Learning in Engineering Science | NCKU + LAiMM | Fall 2026

**Purpose:** a fully worked, self-paced in-class practice with an optional extension. Weeks 11 and 18 are optional examination-readiness materials. These notebooks do not create weekly graded assignments.

**Before running:** choose a Python 3 kernel in Jupyter or a CPU runtime in Colab. Run every cell from top to bottom with Shift+Enter. Setup installs missing free packages. No API key or external dataset download is required.

**Learning target:** complete the calculation, inspect intermediate values, and explain why the protocol supports a limited claim.

**Time:** about 25-45 minutes including reading, prediction, execution, and explain-back.


## 1. Set up and record the environment

Keep the seed fixed for the first run. Record actual outputs and package versions rather than copying a claimed performance number.


In [ ]:
import sys, subprocess, importlib.util, platform, json, random, hashlib
from pathlib import Path
required = {"numpy": "numpy", "matplotlib": "matplotlib", "sklearn": "scikit-learn"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import numpy as np
import matplotlib.pyplot as plt
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Python:", platform.python_version(), "NumPy:", np.__version__, "Seed:", SEED)


**Checkpoint.** The environment cell must finish before any analysis cell. Restart and run all if a variable is missing.


## 2. Construct reference cases before evaluation

Use fixed arithmetic tasks as an inexpensive workflow test. This does not evaluate natural-language understanding.


In [ ]:
cases=[{"id":"a","values":[1,2,3],"expected":2.0},
       {"id":"b","values":[-2,2],"expected":0.0},
       {"id":"c","values":[4,4,4],"expected":4.0}]
def baseline(values): return float(np.mean(values))
def buggy_candidate(values): return float(sum(values)/max(len(values)-1,1))


**Checkpoint.** Expected answers are fixed independently of the candidate implementation.


## 3. Measure correctness with a stated tolerance

An evaluation harness runs identical inputs and compares against the same ground truth. Record per-case failures, not only a summary.


In [ ]:
rows=[]
for case in cases:
    for name,fn in [("baseline",baseline),("candidate",buggy_candidate)]:
        observed=fn(case["values"])
        rows.append({"id":case["id"],"system":name,"observed":observed,"expected":case["expected"],
                     "pass":bool(np.isclose(observed,case["expected"],atol=1e-8))})
for row in rows: print(row)


**Checkpoint.** The intentionally buggy candidate should fail nonzero-mean cases. A denominator error has a concrete mechanism.


## 4. Create an explainable failure report

Do not promote a candidate just because it looks plausible or passes one zero-mean case.


In [ ]:
summary={name:{"passed":sum(r["pass"] for r in rows if r["system"]==name),"total":len(cases)}
         for name in ["baseline","candidate"]}
summary["decision"]="Keep baseline; fix candidate denominator before retesting."
print(json.dumps(summary,indent=2))


**Checkpoint.** Describe why the zero-mean case alone would miss the bug. Add edge cases before expanding the workflow.


## 6. Preserve your evidence

Export a JSON record with a seed, package version, and measured result. Download the artifacts directory before leaving Colab; runtime storage is temporary.


In [ ]:
record={"week":16,"seed":SEED,"numpy":np.__version__,"python":platform.python_version(),"result":summary}
(ARTIFACTS/"week-16-evidence.json").write_text(json.dumps(record,indent=2))
print(json.dumps(record,indent=2))


**Checkpoint.** The evidence file must match the result you actually obtained. A saved screenshot alone does not permit rerunning the calculation.


## 7. Try one change and explain it
1. Predict the effect of one change before editing code. Change one parameter, input, or boundary condition only.
2. Rerun all cells in a fresh kernel. Keep the original evidence record and compare the observed difference.
3. Explain the result using the mechanism, not just the final number.
4. Name one assumption that limits the claim, and a new test that could challenge it.

**Troubleshooting:** if installation fails, restore network access; if a variable is undefined, run earlier cells; if your output differs, compare package versions, seed, and data construction. Do not repeatedly edit code until a preferred score appears.

**Completion criterion:** one executed notebook, one evidence JSON, and a brief claim/evidence/limitation statement.

### Reference documentation
- [NumPy user guide](https://numpy.org/doc/stable/user/)
- [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html)
- [Course repository](https://github.com/CHjonnyYu/MLES-Fall-2026)
